In [1]:
from transformers import BartTokenizer
from datasets import Dataset

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

def preprocess(example):
    # example["input_text"], example["target_text"]
    inputs = tokenizer(
        example["input_text"], 
        truncation=True, 
        padding="max_length", 
        max_length=64
    )
    labels = tokenizer(
        example["target_text"], 
        truncation=True, 
        padding="max_length", 
        max_length=32
    )
    inputs["labels"] = labels["input_ids"]
    return inputs


In [2]:
examples = Dataset.from_list([{ 
        "input_text": "normalize time: 2025-07-29 | next Friday",   
        "target_text": "2025-08-07"  
        },
        { 
        "input_text": "normalize time: 2025-07-29 | two weeks ago",  
        "target_text": "2025-07-15"  
        },
        { 
        "input_text": "normalize time: 2022-01-01 | 3rd quarter of 2021",  
        "target_text": "2021-07-01/2021-09-30"  
        }])

tokenized_train_dataset = examples.map(preprocess, batched=True, remove_columns=["input_text","target_text"])
tokenized_val_dataset = examples.map(preprocess, batched=True, remove_columns=["input_text","target_text"])

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [3]:
from transformers import (
    BartForConditionalGeneration, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq
)

model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")

training_args = Seq2SeqTrainingArguments(
    output_dir="./time_norm_bart",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    num_train_epochs=3,
    predict_with_generate=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    data_collator=data_collator,
)


In [4]:
trainer.train()


Step,Training Loss


d:\GeoTKG\GeoTKG\Lib\site-packages\transformers\modeling_utils.py:3854: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=3, training_loss=11.223803202311197, metrics={'train_runtime': 2.0238, 'train_samples_per_second': 4.447, 'train_steps_per_second': 1.482, 'total_flos': 342976757760.0, 'train_loss': 11.223803202311197, 'epoch': 3.0})

In [6]:
trainer.save_model("./time_norm_bart")
tokenizer.save_pretrained("./time_norm_bart")

('./time_norm_bart\\tokenizer_config.json',
 './time_norm_bart\\special_tokens_map.json',
 './time_norm_bart\\vocab.json',
 './time_norm_bart\\merges.txt',
 './time_norm_bart\\added_tokens.json')

In [7]:
import torch
from transformers import BartForConditionalGeneration, BartTokenizer
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load
tokenizer = BartTokenizer.from_pretrained(os.path.join("time_norm_bart"))
model = BartForConditionalGeneration.from_pretrained(os.path.join("time_norm_bart")).to(device)
model.eval()

def normalize_time(expr: str, ref_date: str) -> str:
    # build your exact input string
    inp = f"normalize time: {ref_date} | {expr}"
    # tokenize
    encoded = tokenizer(
        inp,
        return_tensors="pt",
        truncation=True,
        padding="longest",
    ).to(device)

    # generate
    out_ids = model.generate(
        **encoded,
        max_length=32,
        num_beams=4,       # beam search
        early_stopping=True
    )

    # decode
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

# example
print(normalize_time("next Friday", "2025-07-29"))   # → "2025-08-07"


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


normalize time: 2025-07-29 | next Friday
